# Sketch2Shape AI — complete web app on a Colab GPU

**Your laptop runs only the browser.** Drawing/upload, ControlNet + Stable Diffusion, TripoSR, 3D preview and downloads use the existing application on this Colab machine.

1. Select **Runtime → Change runtime type → T4 GPU** (or another suitable NVIDIA GPU). GPU availability and session limits depend on Colab.
2. Download the development branch source ZIP from GitHub (**Code → Download ZIP**). Run sections **1–3 in order**, or choose **Runtime → Run all**, and upload that ZIP when prompted. Setup can take several minutes; installation output shows the actual progress.
3. Open **Open Sketch2Shape AI** in cell 3. Draw/upload, describe the object, then click **Generate 3D Model**. Download the real GLB from the existing frontend.

Default access is the private Colab port preview, which requires your Colab session. An optional **password-protected temporary Gradio link** is available by explicitly selecting `ACCESS_MODE = "share"` in cell 3, where Colab/network policy permits it. This notebook is not permanent hosting. Keep the runtime connected and download results before it resets.

The notebook reuses the same frontend and backend; no local Windows server, API key, paid inference service, preset output mesh, or laptop GPU is required. Model weights download on the first real generation, not during tests. The integrated pretrained pipeline still needs real GPU acceptance; setup success alone does not establish output quality.

This repository is private. If the direct Colab link requests GitHub access, allow private-repository access in Colab, or extract this notebook from the source ZIP and use **Colab → File → Upload notebook**. The runtime source-upload step does not need GitHub credentials.


## 1 · Get the application source

This repository is **private**, so the default is ZIP upload. From the development branch on GitHub, choose **Code → Download ZIP**, then upload that small source archive when prompted below. No model weights are downloaded to your laptop, and no GitHub token is required in Colab. Run the ZIP helper cell first, then the source cell.

On rerun, the cell reuses the uploaded source. Set `REUSE_UPLOADED_SOURCE = False` to upload a newer ZIP. Existing directories are never overwritten. The optional `git` mode requires repository access from the VM itself; your laptop/Colab GitHub login does not automatically authenticate `git clone`.

In [ ]:
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

def extract_project_zip(archive_file, destination):
    """Normalize Windows ZIP separators, reject unsafe paths, never overwrite."""
    root = Path(destination).resolve()
    if root.exists():
        raise FileExistsError(f"Extraction target already exists: {root}")
    with zipfile.ZipFile(archive_file) as archive:
        entries = archive.infolist()
        if sum(entry.file_size for entry in entries) > 512 * 1024**2:
            raise ValueError("ZIP exceeds 512 MiB unpacked; omit environments/weights/outputs")
        normalized = []
        seen = set()
        for entry in entries:
            name = entry.filename.replace("\\", "/")
            parts = PurePosixPath(name).parts
            if (not parts or name.startswith("/") or ".." in parts
                    or any(":" in part for part in parts)
                    or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError(f"Unsafe ZIP member: {entry.filename!r}")
            target = root.joinpath(*parts)
            if not target.resolve().is_relative_to(root):
                raise ValueError("ZIP path escapes extraction directory")
            if target in seen:
                raise ValueError("ZIP contains duplicate normalized paths")
            seen.add(target)
            normalized.append((entry, target, name.endswith("/")))
        root.mkdir(parents=True)
        for entry, target, is_directory in normalized:
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(entry) as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output)
    matches = list(root.rglob("backend/generator.py"))
    if len(matches) != 1:
        raise ValueError(f"Expected one project backend/generator.py, found {len(matches)}")
    return matches[0].parent.parent.resolve()

In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

try:
    import google.colab
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab.") from exc
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, reconnect, then run this cell again.")
previous_server = globals().get("SERVER")
if previous_server is not None:
    previous_server.stop()
    SERVER = None

SOURCE_MODE = "zip"  # Private-repository default; no GitHub token needed.
REUSE_UPLOADED_SOURCE = True  # Set False to upload an updated source ZIP.
REPOSITORY_URL = "https://github.com/Souravdevsworld/ml-project-techzephyr.git"
SOURCE_REF = "codex/sketch2shape-app"

def checked(command, capture=False):
    return subprocess.run([str(part) for part in command], check=True, text=True, capture_output=capture)

if SOURCE_MODE == "zip":
    existing = globals().get("PROJECT")
    if (REUSE_UPLOADED_SOURCE and existing is not None
            and (Path(existing) / "work/colab/source.json").is_file()
            and (Path(existing) / "scripts/setup_colab.py").is_file()):
        PROJECT = Path(existing)
        print("Reusing uploaded source:", PROJECT)
    else:
        from google.colab import files
        print("Upload the project ZIP downloaded from the codex/sketch2shape-app branch (Code > Download ZIP).")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one source ZIP, without virtual environments or model weights.")
        filename, payload = next(iter(uploaded.items()))
        if not filename.lower().endswith(".zip"):
            raise ValueError("Choose the repository source ZIP.")
        destination = Path("/content/sketch2shape-uploads") / uuid.uuid4().hex
        PROJECT = extract_project_zip(io.BytesIO(payload), destination)
        if not (PROJECT / "scripts/setup_colab.py").is_file():
            raise RuntimeError("This ZIP does not include the Colab app setup. Download the latest development-branch ZIP.")
        identity = {"mode": "zip", "archive_sha256": hashlib.sha256(payload).hexdigest()}
        record = PROJECT / "work/colab/source.json"
        record.parent.mkdir(parents=True, exist_ok=True)
        record.write_text(json.dumps(identity, indent=2), encoding="utf-8")
        print("Uploaded source SHA-256:", identity["archive_sha256"])
        del payload, uploaded
elif SOURCE_MODE == "git":
    # Use only when the VM itself can read the repository, such as a public repo.
    # Colab's GitHub notebook viewer does not pass Git credentials to git clone.
    PROJECT = Path("/content/Sketch2Shape")
    if not PROJECT.exists():
        checked(["git", "clone", REPOSITORY_URL, PROJECT])
    else:
        if not (PROJECT / ".git").exists():
            raise RuntimeError(f"{PROJECT} exists without Git metadata; choose a new project directory.")
        origin = checked(["git", "-C", PROJECT, "remote", "get-url", "origin"], capture=True).stdout.strip()
        if origin != REPOSITORY_URL:
            raise RuntimeError("Existing checkout belongs to a different repository; nothing was overwritten.")
        if checked(["git", "-C", PROJECT, "status", "--porcelain"], capture=True).stdout.strip():
            raise RuntimeError("Existing source has local edits. Preserve them before updating; no reset was performed.")
    checked(["git", "-C", PROJECT, "fetch", "origin", SOURCE_REF])
    checked(["git", "-C", PROJECT, "switch", "--detach", "FETCH_HEAD"])
    print("Application source:", checked(["git", "-C", PROJECT, "rev-parse", "HEAD"], capture=True).stdout.strip())
else:
    raise ValueError("SOURCE_MODE must be zip or git.")
print("Application directory:", PROJECT)

## 2 · Prepare the GPU workers and frontend

The setup keeps **Gradio** and the **ML stack** in separate environments. It reuses Colab's working CUDA PyTorch when available without replacing the notebook kernel packages. If needed, it installs an isolated Python 3.12 runtime and CUDA PyTorch. Model extraction uses real scikit-image marching cubes, avoiding a custom CUDA extension build.

The checks include a small CUDA operation, dependency imports and the upstream TripoSR entry point. They do **not** load model weights or run the full pipeline. Stop and inspect the first error if setup fails.

In [ ]:
active_server = globals().get("SERVER")
if active_server is not None and active_server.process is not None and active_server.process.poll() is None:
    raise RuntimeError("Stop the running app before rerunning dependency setup.")
CACHE_ROOT = Path("/content/sketch2shape-cache")  # Reuse model downloads across source ZIP updates.
checked([sys.executable, PROJECT / "scripts" / "setup_colab.py", "--cache-root", CACHE_ROOT])
RUNTIME_FILE = PROJECT / "work" / "colab" / "runtime.json"
runtime = json.loads(RUNTIME_FILE.read_text(encoding="utf-8"))
print("GPU:", runtime["hardware"]["gpu"])
print("VRAM (GiB):", round(runtime["hardware"]["vram_bytes"] / 1024**3, 1))
print("Setup completed. Open the app in the next cell; models download when you generate.")

## 3 · Open the existing frontend

- **`proxy` (default):** private preview through your Colab session. Use the embedded view if a new tab cannot open.
- **`share`:** requests a temporary external Gradio link protected by a password you enter without echo. Login username is `sketch2shape`. Use it only where the platform permits sharing. No public link is requested in proxy mode.

The normal T4 profile uses 512px refinement, CFG 7, 20 steps and mesh resolution 128. `LOW_MEMORY = True` starts with the smaller 256px/CFG 1 profile if needed. Both modes use one GPU job at a time and sequential model workers. Start with a single object and a clear description; leaving the description empty uses a neutral prompt without object recognition.

In [ ]:
import getpass
from html import escape
from IPython.display import HTML, display
from google.colab import output

ACCESS_MODE = "proxy"  # Change to "share" only if you want a password-protected external link.
LOW_MEMORY = False     # Normal T4 profile; set True for the smaller/slower profile.
PORT = 7860

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import importlib.util
spec = importlib.util.spec_from_file_location("sketch2shape_colab_runtime", PROJECT / "scripts/colab_runtime.py")
colab_runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_runtime)
AppServer = colab_runtime.AppServer
if Path(runtime["project"]).resolve() != PROJECT.resolve():
    raise RuntimeError("Rerun setup for the newly uploaded source before launching.")

# Stop only the app process started by an earlier execution of this cell.
previous_server = globals().get("SERVER")
if previous_server is not None:
    previous_server.stop()
password = getpass.getpass("Choose an app password (at least 12 characters): ") if ACCESS_MODE == "share" else None
SERVER = AppServer(runtime, port=PORT, access=ACCESS_MODE, password=password, low_memory=LOW_MEMORY)
try:
    print("Starting the frontend on the Colab host...")
    SERVER.start()
finally:
    # The server needs its inherited environment; the notebook does not retain
    # a second explicit password variable for later printing.
    password = None
if ACCESS_MODE == "share":
    print("Waiting for the requested password-protected temporary link...")
    APP_URL = SERVER.share_url()
    print("Login username: sketch2shape. Use the password you entered above.")
else:
    APP_URL = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
display(HTML(f'<p><a href="{escape(APP_URL, quote=True)}" target="_blank" rel="noopener">Open Sketch2Shape AI</a></p>'))
if ACCESS_MODE == "proxy":
    output.serve_kernel_port_as_iframe(PORT, height=1000)
print("The app stays running while you use the browser. Server log:", SERVER.log)

## Generate and inspect

Use the frontend opened above:

1. Choose **Sketch to 3D**, draw/upload a fresh sketch and optionally describe it.
2. Click **Generate 3D Model**. First use downloads actual ControlNet, Stable Diffusion, TripoSR and optional background-removal weights.
3. Inspect the conditioning map, refined image and interactive 3D model. A model is shown only after GLB validation.
4. Download the GLB and PNG/metadata files from the frontend. The preview and GLB download use the same validated model.

**Image to 3D** uses an object photo and bypasses sketch refinement. Do not expect a single view to determine hidden geometry accurately. Check recognizable shape and sketch fidelity visually; structural validation is not a quality guarantee.

Your Windows Python/GPU environment is not used. Files live in this Colab runtime; local Windows paths cannot be passed to its workers. Upload through the frontend instead.

## 4 · Optional: download a matched input/intermediate/model bundle

The frontend already offers individual downloads. To save a ZIP of the **most recent successful job**, set the switch below to `True` and run the cell after generation. This validates the real GLB again and includes its input, intermediate images and metadata. It may refer to an earlier success if the latest attempt failed; check the saved input/job ID. Nothing is generated or substituted by this export.

In [ ]:
DOWNLOAD_LAST_SUCCESS = False  # Set True and rerun this cell after a successful generation.
if DOWNLOAD_LAST_SUCCESS:
    from google.colab import files
    export = checked([runtime["ui_python"], PROJECT / "scripts" / "colab_runtime.py", "--export-project", PROJECT], capture=True)
    archive = json.loads(export.stdout)["archive"]
    if archive is None:
        print("No successful job yet. Generate in the frontend, then rerun this cell.")
    else:
        print("Downloading completed job bundle:", Path(archive).name)
        files.download(archive)
else:
    print("Use the app downloads, or set DOWNLOAD_LAST_SUCCESS = True after generation.")

## 5 · Optional: inspect logs or stop the app

Keep `SHOW_LOGS` and `STOP_APP` false during **Run all**, so the application remains available. Failed requests clear the frontend results and identify their job log. The helper below displays the latest stage log only when requested. Model/port failures should be fixed from their actual traceback rather than repeated parameter guesses.

In [ ]:
SHOW_LOGS = False
STOP_APP = False  # Set True and run this cell only when finished.
if SHOW_LOGS:
    logs = list((PROJECT / "outputs" / "jobs").glob("*/*.log"))
    if logs:
        latest = max(logs, key=lambda path: path.stat().st_mtime)
        print("Latest worker log:", latest)
        print("\n".join(latest.read_text(encoding="utf-8", errors="replace").splitlines()[-100:]))
    elif globals().get("SERVER") is not None and SERVER.log.is_file():
        print("\n".join(SERVER.log.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]))
    else:
        print("No app/worker log yet.")
if STOP_APP and globals().get("SERVER") is not None:
    SERVER.stop()
    print("App stopped. Download any remaining files before disconnecting the runtime.")
else:
    print("The app remains running. Keep Colab connected while using it.")

### Session limits and acceptance

Colab GPU availability, RAM, quotas and external-link behavior vary. This creates a temporary application, not guaranteed permanent hosting. The Colab proxy and sharing options require the platform's normal policies and networking to permit them; no network restrictions are bypassed. If a shared link cannot be established, choose proxy mode and rerun cell 3.

Download outputs before disconnecting. Optional Google Drive persistence can be arranged by saving the completed job ZIP yourself; the notebook does not automatically mount or write to Drive.

Full real GPU acceptance is still required: fresh chair sketch → actual refined PNG → validated/downloaded GLB → reopen and inspect. Hardware/timings are recorded by the existing workers. Offline/synthetic tests or a successful setup do not count as pretrained inference success.

Models: Scribble ControlNet 1.1, Stable Diffusion 1.5 and TripoSR. Review their model cards/licenses in the repository README. Original preprocessing, neural model inference and mesh validation remain in use.